In [9]:
import torch
import torchaudio

In [ ]:
# Ambil modelnya
bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
model = bundle.get_model()

: 

In [ ]:
# Wave, sample rate dari audio file
wave, sample_rate = torchaudio.load("speech.wav")

RuntimeError: Numpy is not available

: 

In [ ]:
# Pre-process wave dan sample ratenya
if sample_rate != bundle.sample_rate:
    wave = torchaudio.functional.resample(wave, sample_rate, bundle.sample_rate)

NameError: name 'sample_rate' is not defined

: 

In [ ]:
with torch.inference_mode():
    emission, _ = model(wave)
print(emission)

: 

In [ ]:
# Ubah jadi teks
# _labels = ('|', 'E', 'T', 'A', 'O', 'N', 'I', 'H', 'S', 'R')
class GreedyCTCDecoder(torch.nn.Module):
    # Constructor
    def __init__(self, labels, blank=0):
        super().__init__()
        self.labels = labels
        self.blank = blank

    def forward(self, emission: torch.Tensor) -> str:
        indices = torch.argmax(emission, dim=-1)
        # print(indices)
        indices = torch.unique_consecutive(indices, dim=-1)
        # print(indices)
        indices = [i for i in indices if i != self.blank]
        # print(indices)
        return "".join(self.labels[i] for i in indices)

: 

In [ ]:
decoder = GreedyCTCDecoder(labels=bundle.get_labels())
sentence = decoder(emission[0])
print(sentence)

: 

: 